In [1]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244081109

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")

[Conda 环境] py311
[学号] 20244081109
[机器指纹] 21935ea21d1fabf1
[RUN_ID] 20244081109-21935ea21d1fabf1


# 第一部分：Python 正则表达式练习
## 1. 检查字符串是否仅包含英文字母 a-z、A-Z 和数字 0-9

思路：字符类 `[A-Za-z0-9]` 表示“一个字母或数字”，`+` 要求至少出现一次，`fullmatch` 要求整个字符串都被覆盖。空串不含任何字符，这里约定返回 False。

In [2]:
import re

def is_alnum_only(s):
    return re.fullmatch(r"[A-Za-z0-9]+", s) is not None

# 测试
assert is_alnum_only("ABCabc123") is True
assert is_alnum_only("2026") is True
assert is_alnum_only("abc-123") is False   # 含连字符
assert is_alnum_only("hello world") is False  # 含空格
assert is_alnum_only("") is False          # 空串
assert is_alnum_only("你好123") is False    # 非英文字母

print("示例输出：")
for s in ["ABCabc123", "2026", "abc-123", "hello world", "", "你好123"]:
    print(f"  {s!r} -> {is_alnum_only(s)}")

示例输出：
  'ABCabc123' -> True
  '2026' -> True
  'abc-123' -> False
  'hello world' -> False
  '' -> False
  '你好123' -> False


## 2. 匹配一个 a 后面跟着零个或多个 b

思路：`b*` 表示 b 出现 0 次或多次，所以 "a" 本身也算匹配；用 `fullmatch` 排除后面混入其他字符的情况。

In [3]:
def match_a_b0(s):
    return re.fullmatch(r"ab*", s) is not None

assert match_a_b0("a") is True      # 0 个 b
assert match_a_b0("ab") is True
assert match_a_b0("abbbbb") is True
assert match_a_b0("b") is False     # 缺少 a
assert match_a_b0("abba") is False  # 结尾多了 a
assert match_a_b0("ac") is False

print("示例输出：")
for s in ["a", "ab", "abbbbb", "b", "abba", "ac"]:
    print(f"  {s!r} -> {match_a_b0(s)}")

示例输出：
  'a' -> True
  'ab' -> True
  'abbbbb' -> True
  'b' -> False
  'abba' -> False
  'ac' -> False


## 3. 匹配一个 a 后面跟着一个或多个 b

思路：与第 2 题只差一个量词——`b+` 要求 b 至少出现 1 次，因此 "a" 不再匹配。

In [4]:
def match_a_b1(s):
    return re.fullmatch(r"ab+", s) is not None

assert match_a_b1("ab") is True
assert match_a_b1("abbbbb") is True
assert match_a_b1("a") is False     # 0 个 b，不满足 +
assert match_a_b1("b") is False
assert match_a_b1("abab") is False

print("示例输出：")
for s in ["ab", "abbbbb", "a", "b", "abab"]:
    print(f"  {s!r} -> {match_a_b1(s)}")

示例输出：
  'ab' -> True
  'abbbbb' -> True
  'a' -> False
  'b' -> False
  'abab' -> False


## 4. 匹配以 a 开头、以 b 结尾，中间任意字符

思路：`a.*b`——`.` 匹配除换行外的任意字符，`*` 表示任意个数（可以是 0 个，所以 "ab" 也匹配）。若希望 `.` 连换行也匹配，可加 `re.DOTALL` 标志。

In [5]:
def match_a_start_b_end(s):
    return re.fullmatch(r"a.*b", s) is not None

assert match_a_start_b_end("ab") is True
assert match_a_start_b_end("axxb") is True
assert match_a_start_b_end("a123!b") is True
assert match_a_start_b_end("a") is False    # 不以 b 结尾
assert match_a_start_b_end("bxa") is False  # 不以 a 开头
assert match_a_start_b_end("ba") is False

print("示例输出：")
for s in ["ab", "axxb", "a123!b", "a", "bxa"]:
    print(f"  {s!r} -> {match_a_start_b_end(s)}")

示例输出：
  'ab' -> True
  'axxb' -> True
  'a123!b' -> True
  'a' -> False
  'bxa' -> False


## 5. 删除 IP 地址各段的前导零

思路：`\b0+(\d)` —— `\b` 是单词边界（IP 中的 `.` 是非单词字符，所以每段开头都是一个边界），`0+` 吃掉前导零，捕获组 `(\d)` 保住紧随其后的第一个有效数字，替换为 `\1`。
某段全为 0 时（如 `000`），`0+` 最多只能吃掉 2 个 0，最后一个 0 被 `(\d)` 捕获保留，天然满足“全零保留一个 0”。

In [6]:
def strip_ip_leading_zeros(s):
    return re.sub(r"\b0+(\d)", r"\1", s)

assert strip_ip_leading_zeros("192.168.001.010") == "192.168.1.10"
assert strip_ip_leading_zeros("000.000.000.000") == "0.0.0.0"
assert strip_ip_leading_zeros("010.001.000.199") == "10.1.0.199"
assert strip_ip_leading_zeros("127.0.0.1") == "127.0.0.1"  # 无前导零则不变

print("示例输出：")
for s in ["192.168.001.010", "000.000.000.000", "010.001.000.199", "127.0.0.1"]:
    print(f"  {s} -> {strip_ip_leading_zeros(s)}")

示例输出：
  192.168.001.010 -> 192.168.1.10
  000.000.000.000 -> 0.0.0.0
  010.001.000.199 -> 10.1.0.199
  127.0.0.1 -> 127.0.0.1


## 6. 空格与下划线互换

思路：不能先 `replace(" ", "_")` 再 `replace("_", " ")`——第一步产生的下划线会在第二步被换回空格，等于没换。正确做法是**一次遍历同时决定**：用 `re.sub` 匹配 `[ _]`，替换函数根据匹配到的是空格还是下划线返回对方。

In [7]:
def swap_space_underscore(s):
    return re.sub(r"[ _]", lambda m: "_" if m.group() == " " else " ", s)

assert swap_space_underscore("a b_c") == "a_b c"
assert swap_space_underscore("hello world_foo bar") == "hello_world foo_bar"
assert swap_space_underscore("nochange") == "nochange"
assert swap_space_underscore("  __  ") == "__  __"

print("示例输出：")
for s in ["a b_c", "hello world_foo bar", "  __  "]:
    print(f"  {s!r} -> {swap_space_underscore(s)!r}")

示例输出：
  'a b_c' -> 'a_b c'
  'hello world_foo bar' -> 'hello_world foo_bar'
  '  __  ' -> '__  __'


## 7. yyyy-mm-dd 转 dd-mm-yyyy

思路：三个捕获组分别抓年、月、日，替换串 `\3-\2-\1` 把组序倒过来。`\b` 保证不会误伤更长数字串的一部分。

In [8]:
def convert_date(s):
    return re.sub(r"\b(\d{4})-(\d{2})-(\d{2})\b", r"\3-\2-\1", s)

assert convert_date("2026-11-25") == "25-11-2026"
assert convert_date("1999-01-01") == "01-01-1999"
assert convert_date("生于 2000-05-20，毕业 2022-06-30") == "生于 20-05-2000，毕业 30-06-2022"
assert convert_date("2026-1-5") == "2026-1-5"  # 非两位月日，不处理

print("示例输出：")
for s in ["2026-11-25", "1999-01-01", "生于 2000-05-20，毕业 2022-06-30"]:
    print(f"  {s} -> {convert_date(s)}")

示例输出：
  2026-11-25 -> 25-11-2026
  1999-01-01 -> 01-01-1999
  生于 2000-05-20，毕业 2022-06-30 -> 生于 20-05-2000，毕业 30-06-2022


## 8. 找出所有以 a 或 e 开头的单词

思路：`\b[aeAE][A-Za-z]*` —— `\b` 定位单词开头，`[aeAE]` 限定首字母（含大写），后面跟任意个字母。`findall` 返回全部匹配。

In [9]:
def find_words_a_e(s):
    return re.findall(r"\b[aeAE][A-Za-z]*\b", s)

assert find_words_a_e("apple elephant banana Egg orange ant") == ["apple", "elephant", "Egg", "ant"]
assert find_words_a_e("An apple a day") == ["An", "apple", "a"]
assert find_words_a_e("xyz banana") == []

print("示例输出：")
for s in ["apple elephant banana Egg orange ant", "An apple a day", "xyz banana"]:
    print(f"  {s!r}\n    -> {find_words_a_e(s)}")

示例输出：
  'apple elephant banana Egg orange ant'
    -> ['apple', 'elephant', 'Egg', 'ant']
  'An apple a day'
    -> ['An', 'apple', 'a']
  'xyz banana'
    -> []


## 9. 将 Road 缩写为 Rd.

思路：`\bRoad\b` 只匹配独立单词 Road，`\b` 保证 Roadside、Crossroad 里的 Road 不会被误替换。

In [10]:
def abbreviate_road(s):
    return re.sub(r"\bRoad\b", "Rd.", s)

assert abbreviate_road("Jiangnan Road") == "Jiangnan Rd."
assert abbreviate_road("Road to the North Road") == "Rd. to the North Rd."
assert abbreviate_road("Roadside Crossroad") == "Roadside Crossroad"  # 不误伤

print("示例输出：")
for s in ["Jiangnan Road", "Road to the North Road", "Roadside Crossroad"]:
    print(f"  {s} -> {abbreviate_road(s)}")

示例输出：
  Jiangnan Road -> Jiangnan Rd.
  Road to the North Road -> Rd. to the North Rd.
  Roadside Crossroad -> Roadside Crossroad


## 10. 找出所有长度至少 4 个字符的单词

思路：`\b\w{4,}\b` —— `{4,}` 表示至少重复 4 次。注意 `\w` 含字母、数字、下划线；若只想统计纯字母单词，可改用 `[A-Za-z]{4,}`。

In [11]:
def find_long_words(s):
    return re.findall(r"\b\w{4,}\b", s)

assert find_long_words("The quick brown fox jumps over a lazy dog") == ["quick", "brown", "jumps", "over", "lazy"]
assert find_long_words("a bb ccc dddd eeeee") == ["dddd", "eeeee"]
assert find_long_words("no long words here") == ["long", "words", "here"]

print("示例输出：")
for s in ["The quick brown fox jumps over a lazy dog", "a bb ccc dddd eeeee"]:
    print(f"  {s!r}\n    -> {find_long_words(s)}")

示例输出：
  'The quick brown fox jumps over a lazy dog'
    -> ['quick', 'brown', 'jumps', 'over', 'lazy']
  'a bb ccc dddd eeeee'
    -> ['dddd', 'eeeee']


## 11. 删除除字母和数字之外的所有字符

思路：`[^A-Za-z0-9]` 是**取反**字符类，匹配一切非英文字母/数字的字符（标点、空格、中文等），替换为空串即删除。

In [12]:
def remove_non_alnum(s):
    return re.sub(r"[^A-Za-z0-9]", "", s)

assert remove_non_alnum("Hello, World! 123") == "HelloWorld123"
assert remove_non_alnum("a-b_c@d#e$f") == "abcdef"
assert remove_non_alnum("你好abc，123！") == "abc123"
assert remove_non_alnum("!!!") == ""

print("示例输出：")
for s in ["Hello, World! 123", "a-b_c@d#e$f", "你好abc，123！"]:
    print(f"  {s!r} -> {remove_non_alnum(s)!r}")

示例输出：


  'Hello, World! 123' -> 'HelloWorld123'
  'a-b_c@d#e$f' -> 'abcdef'
  '你好abc，123！' -> 'abc123'


## 12. 删除括号及括号内的内容

思路：`\s*\([^)]*\)` —— `\s*` 顺带吃掉括号前的空白（避免留下尾随空格），`\(` 和 `\)` 是转义后的字面括号，`[^)]*` 匹配括号内任意非右括号字符。删除后再 `strip()` 清理两端。

In [13]:
def remove_parentheses(s):
    return re.sub(r"\s*\([^)]*\)", "", s).strip()

assert remove_parentheses("example (.com)") == "example"
assert remove_parentheses("w3resource") == "w3resource"
assert remove_parentheses("github (.com)") == "github"
assert remove_parentheses("stackoverflow (.com)") == "stackoverflow"
assert remove_parentheses("a (x) b (y)") == "a b"  # 括号前的空格一并删除

print("示例输出：")
for s in ["example (.com)", "w3resource", "github (.com)", "stackoverflow (.com)"]:
    print(f"  {s!r} -> {remove_parentheses(s)!r}")

示例输出：


  'example (.com)' -> 'example'
  'w3resource' -> 'w3resource'
  'github (.com)' -> 'github'
  'stackoverflow (.com)' -> 'stackoverflow'


# 第二部分：从网页检索气候数据

**实际使用的页面与访问日期（2026-11-25）**：英文维基百科站点 `en.wikipedia.org`，条目路径 `/wiki/城市名`，城市为 `Harbin`、`Madison,_Wisconsin`、`Boston`、`Los_Angeles`（完整 URL 见下方 `page_url()`）。

**访问情况记录**：本机直连时四个页面均出现 `requests.exceptions.ConnectTimeout`（TCP 连接超时）。这不是“页面没有气候表格”，而是网络不可达；改用本机代理 `127.0.0.1:7897` 后全部返回 HTTP 200。因此 `fetch_html()` 的策略是：先尝试直连（短超时），失败后自动走本地代理重试；两条路都失败才抛异常。

**实测到的结构差异**（用于设计测试）：
- Harbin：页面有 2 个气候表格；标题在表格第一行的 `<th>` 中（无 `<caption>`）；数据行名带 `scope="row"`。
- Madison：结构同 Harbin，单位为华氏。
- Boston：气候表格（海水温度表）的 `<th>` **完全没有 scope 属性**，需要回退策略。
- Los Angeles：页面可正常访问，但**没有**标题以 “Climate data for” 开头的表格 → 应返回 `None`。

In [14]:
import re
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag

SITE = "https://en.wikipedia.org"   # 英文维基百科站点
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) PythonCoursework/1.0"}
LOCAL_PROXY = {"http": "http://127.0.0.1:7897", "https": "http://127.0.0.1:7897"}

def page_url(city):
    '''由城市名拼接条目 URL，例如 page_url("Harbin")'''
    return f"{SITE}/wiki/{city}"

def fetch_html(url):
    '''获取页面 HTML。先直连（短超时），网络不通则改走本地代理；
    HTTP 错误（403/404/429 等）由 raise_for_status() 抛出异常。'''
    try:
        resp = requests.get(url, headers=HEADERS, timeout=8)
    except requests.exceptions.RequestException:
        resp = requests.get(url, headers=HEADERS, proxies=LOCAL_PROXY, timeout=30)
    resp.raise_for_status()
    return resp.text

def normalize(text):
    '''get_text() 的结果常含换行和连续空白，合并为单个空格并去首尾'''
    return re.sub(r"\s+", " ", text).strip()

print("工具函数就绪")

工具函数就绪


## 气候题 1：读取 Harbin 页面并观察气候表格结构

思路：用 `requests` 取 HTML，`BeautifulSoup(html, "html.parser")` 解析；遍历所有 `<table>`，取第一个 `<th>` 的文本（合并空白后）判断是否以 “Climate data for” 开头。找到后观察其**直接子节点**（`table.children` 中过滤掉空白字符串）与 `<tr>` 的组成。

In [15]:
html = fetch_html(page_url("Harbin"))
soup = BeautifulSoup(html, "html.parser")

climate_table = None
for t in soup.find_all("table"):
    th = t.find("th")
    title = normalize(th.get_text()) if th else ""
    if title.startswith("Climate data for"):
        climate_table = t
        break

assert climate_table is not None
print("表格标题:", title)

children = [c.name for c in climate_table.children if c.name is not None]
print("直接子节点数量:", len(children), "| 标签名:", children)

rows = climate_table.find_all("tr")
print("tr 行数:", len(rows))
for i, tr in enumerate(rows[:4]):
    ths, tds = tr.find_all("th"), tr.find_all("td")
    first = normalize(ths[0].get_text())[:60] if ths else None
    print(f"tr[{i}]: th x {len(ths)}, td x {len(tds)} | 首个 th: {first}")
print("最后一行文本:", normalize(rows[-1].get_text())[:80])

表格标题: Climate data for Harbin, elevation 118 m (387 ft), (1991–2020 normals, extremes 1951–present)
直接子节点数量: 1 | 标签名: ['tbody']
tr 行数: 18
tr[0]: th x 1, td x 0 | 首个 th: Climate data for Harbin, elevation 118 m (387 ft), (1991–202
tr[1]: th x 14, td x 0 | 首个 th: Month
tr[2]: th x 1, td x 13 | 首个 th: Record high °C (°F)
tr[3]: th x 1, td x 13 | 首个 th: Mean maximum °C (°F)
最后一行文本: Source 2: China Meteorological Administration,[91][92][93] Weather China[94] ext


**结构观察结论**（以实际运行输出为准）：
- `<table>` 的直接子节点只有 **1 个 `<tbody>`**（`html.parser` 解析出的表格行都包在 tbody 里），class 为 `wikitable mw-collapsible`。
- 第 1 个 `<tr>` 是标题行：单个带 `colspan` 的 `<th>`，文本以 “Climate data for Harbin...” 开头，**没有 `<caption>`**。
- 第 2 个 `<tr>` 是月份表头：`<th>Month</th>` + Jan–Dec + Year，各月 `<th scope="col">`。
- 之后每个数据行 = 1 个 `<th scope="row">`（行名，如 “Record high °C (°F)”）+ 若干 `<td>` 数值。
- 末尾是数据来源行（“Source: ...”），不属于数据行。

## 气候题 2：retrieve_climate_table(s)

思路：
1. `fetch_html(s)` 取页面——网络错误 / HTTP 错误在这里**直接抛出**，不吞掉；
2. 遍历每个 `<table>`，候选标题取 `<caption>` 文本和第一个 `<th>` 文本（两种位置都见过），`normalize` 合并空白后判断前缀；
3. 命中即返回该 `Tag`；遍历完没有命中返回 `None`。多个气候表格时返回第一个（题目允许任意一个）。

In [16]:
def retrieve_climate_table(s):
    '''s 为城市页面 URL 字符串。
    找到气候数据表格 -> 返回 BeautifulSoup Tag；
    页面正常但无此类表格 -> 返回 None；
    网络/HTTP 错误 -> 抛出异常。'''
    html = fetch_html(s)
    soup = BeautifulSoup(html, "html.parser")
    for table in soup.find_all("table"):
        candidates = []
        cap = table.find("caption")
        if cap is not None:
            candidates.append(normalize(cap.get_text()))
        th = table.find("th")
        if th is not None:
            candidates.append(normalize(th.get_text()))
        if any(t.startswith("Climate data for") for t in candidates):
            return table
    return None

# 测试 1：三个有气候表格的城市（Boston 结构与 Harbin 不同：无 scope 属性）
for city in ["Harbin", "Madison,_Wisconsin", "Boston"]:
    t = retrieve_climate_table(page_url(city))
    assert isinstance(t, Tag), city
    print(city, "->", normalize(t.find("th").get_text())[:70])

# 测试 2：页面正常但没有气候表格 -> None
la = retrieve_climate_table(page_url("Los_Angeles"))
assert la is None
print("Los_Angeles -> None（页面正常，无 Climate data for 表格）")

# 测试 3：HTTP 错误应抛异常（故意访问不存在的条目，预期 404）
try:
    retrieve_climate_table(page_url("Hw04_No_Such_Page_20261125"))
except requests.HTTPError as e:
    print("404 测试:", type(e).__name__, "状态码", e.response.status_code)
else:
    raise AssertionError("应当抛出 HTTPError")

Harbin -> Climate data for Harbin, elevation 118 m (387 ft), (1991–2020 normals,


Madison,_Wisconsin -> Climate data for Madison, Wisconsin (Dane County Regional Airport), 19


Boston -> Climate data for Boston, Massachusetts


Los_Angeles -> None（页面正常，无 Climate data for 表格）


404 测试: HTTPError 状态码 404


## 气候题 3：list_climate_table_row_names(s)

思路（结合 `<tr>` 结构与 `<th>` 位置）：
1. 复用 `retrieve_climate_table`；返回 `None` 时本函数也返回 `None`；
2. 逐行找行名：**优先** `th[scope="row"]`（Harbin/Madison 有）；没有该属性时（Boston）**回退**为该行第一个 `<th>`，但要求该行含 `<td>` 数据单元格且该 `<th>` 不带 `colspan`——这样标题行、月份表头行、来源行都会被自然排除；
3. 再按文本兜底过滤：“Month”、“Climate data for...” 开头、“Source” 开头；
4. 每项都经 `normalize` 处理，保证是去首尾空白的纯 `str`。

In [17]:
def list_climate_table_row_names(s):
    '''s 为城市页面 URL。找到表格 -> 行名列表（纯字符串）；找不到 -> None。'''
    table = retrieve_climate_table(s)
    if table is None:
        return None
    names = []
    for tr in table.find_all("tr"):
        th = tr.find("th", scope="row")
        if th is None:
            # 回退策略：行内第一个 th，且该行必须有数据单元格、th 不跨列
            if not tr.find_all("td"):
                continue
            ths = tr.find_all("th")
            if not ths:
                continue
            th = ths[0]
            if th.get("colspan"):
                continue
        name = normalize(th.get_text())
        if (not name or name == "Month"
                or name.startswith("Climate data for")
                or name.startswith("Source")):
            continue
        names.append(name)
    return names

# 测试 1：Harbin —— 检查类型、去空白、不含标题/月份/来源行
harbin = list_climate_table_row_names(page_url("Harbin"))
assert isinstance(harbin, list)
assert all(isinstance(x, str) and x == x.strip() for x in harbin)
assert "Record high °C (°F)" in harbin
assert "Month" not in harbin
assert not any(x.startswith(("Climate data for", "Source")) for x in harbin)
print(f"Harbin 行名（{len(harbin)} 项）:")
for n in harbin:
    print("  -", n)

# 测试 2：Madison —— 结构相同、单位不同
madison = list_climate_table_row_names(page_url("Madison,_Wisconsin"))
assert "Record high °F (°C)" in madison and "Month" not in madison
print(f"Madison 行名（{len(madison)} 项）:", madison[:3], "...")

# 测试 3：Boston —— 无 scope 属性，验证回退策略
boston = list_climate_table_row_names(page_url("Boston"))
assert "Average sea temperature °F (°C)" in boston
assert "Month" not in boston
print("Boston 行名:", boston)

# 测试 4：无气候表格 -> None
assert list_climate_table_row_names(page_url("Los_Angeles")) is None
print("Los_Angeles -> None")

Harbin 行名（14 项）:
  - Record high °C (°F)
  - Mean maximum °C (°F)
  - Mean daily maximum °C (°F)
  - Daily mean °C (°F)
  - Mean daily minimum °C (°F)
  - Mean minimum °C (°F)
  - Record low °C (°F)
  - Average precipitation mm (inches)
  - Average precipitation days (≥ 0.1 mm)
  - Average snowy days
  - Average relative humidity (%)
  - Mean monthly sunshine hours
  - Percentage possible sunshine
  - Average ultraviolet index


Madison 行名（15 项）: ['Record high °F (°C)', 'Mean maximum °F (°C)', 'Mean daily maximum °F (°C)'] ...


Boston 行名: ['Average sea temperature °F (°C)']


Los_Angeles -> None
